In [ ]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.797,io.delta:delta-spark_2.12:3.3.2")
    .set("spark.executor.heartbeatInterval", "300000")
    .set("spark.network.timeout", "400000")
    .set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    .set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    .set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.delta.enableFastS3AListFrom", "true")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

# configure the SparkSession with the configure_spark_with_delta_pip() utility function in Delta Lake:
builder = SparkSession.builder.appName("spark-deltalake-timetravel").master("local[*]").config(conf=sparkConf)
spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4"]).getOrCreate()

#
spark.sparkContext.setLogLevel('ERROR')
spark

#
# 
#
def display(df):
    df.show(truncate=False)

In [ ]:
%load_ext sql

In [ ]:
%sql spark

##### Delete Existing Delta Table

In [ ]:
%%sql

DROP TABLE IF EXISTS delta.`/deltalake/invoices_ttv`;

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://defaultfs/deltalake/invoices_ttv --recursive

##### Create Delta Table

In [ ]:
df = spark.read.format("parquet").load("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet")
df.printSchema()

#
display(df)

# Save as delta table into Minio S3
df.write.format('delta').save('/deltalake/invoices_ttv')

In [ ]:
%%sql

CREATE OR REPLACE TABLE delta.`/deltalake/invoices_ttv` USING  DELTA
AS SELECT * FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`;

In [ ]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_ttv`
LIMIT 5;

In [ ]:
%%sql

DELETE FROM delta.`/deltalake/invoices_ttv`
WHERE customer_id = 1;

In [ ]:
%%sql

UPDATE delta.`/deltalake/invoices_ttv`
SET quantity = 25
WHERE customer_id = 5; 

In [ ]:
%%sql

INSERT INTO delta.`/deltalake/invoices_ttv`
SELECT * FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_101_200.parquet`

In [ ]:
%%sql


SELECT COUNT(*) FROM delta.`/deltalake/invoices_ttv`
WHERE customer_id > 100; 

In [ ]:
%%sql

DESCRIBE HISTORY delta.`/deltalake/invoices_ttv`;

In [ ]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_ttv` VERSION AS OF 0
WHERE customer_id = 1;

In [ ]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_ttv` TIMESTAMP AS OF '2026-03-31T07:45:09.000+00:00'
WHERE customer_id = 1;

In [ ]:
%%sql

RESTORE TABLE delta.`/deltalake/invoices_ttv` TO VERSION AS OF 0; 
-- RESTORE TABLE delta.`/deltalake/invoices_ttv` TO TIMESTAMP AS OF '2025-02-02T07:08:09.000+00:00';

In [ ]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_ttv`
WHERE customer_id = 1;

In [ ]:
%%sql

DESCRIBE HISTORY delta.`/deltalake/invoices_ttv`;

In [0]:
-- 3	2025-02-02T07:09:57.000+00:00
--   2025-02-02T07:09:35.000+00:00
-- 2 2025-02-02T07:09:31.000+00:00

In [ ]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_ttv` TIMESTAMP AS OF '2026-03-31 13:19:57' -- version 2
WHERE customer_id = 5;  

In [ ]:
%%sql

SELECT * FROM delta.`/deltalake/invoices_ttv` -- version 0
WHERE customer_id = 5;  

In [ ]:

from pyspark.sql.functions import col
df = spark.read.option("versionAsOf", "1").table("delta.`/deltalake/invoices_ttv`")
display(df.filter(col("customer_id") == 1))

In [ ]:

df = spark.read.option("versionAsOf", "2").table("delta.`/deltalake/invoices_ttv`")
display(df.filter(col("customer_id") == 5))

In [ ]:

df = spark.read.option("timestampAsOf", "2026-03-31 13:19:57").table("delta.`/deltalake/invoices_ttv`")
display(df.filter(col("customer_id") == 5))